# Top+Side View Volume Estimation
# Feature Extractor with Trainable Last Stage + POV Fusion + Regression Head

This notebook trains and evaluates different combinations of feature extractors with unfrozen last stage, and regression heads on the top+side dataset. Uses concat fusion strategy.


In [1]:
import datetime
import json
import warnings
from src.training_and_evaluation import *
from src.trainable_pipelines import *

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

/home/lenka-hake/Documents/CV/ComputerVisionProject/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## Setup

In [2]:
LAST_LAYER_OUTER_SPLITS = 3
LAST_LAYER_INNER_SPLITS = 3

LAST_LAYER_BACKBONE_NAMES = [
    "densenet121",
    # "vit_b_16"
]

LAST_LAYER_FUSION_NAMES = [
    "concat"
]

LAST_LAYER_REGRESSION_MODEL_CONFIGS = {
    "linear": list(ParameterGrid({
        "lr_backbone": [1e-4],
        "lr_head": [1e-3],
        "weight_decay": [1e-3],
        # "lr_head": [1e-3, 3e-4],
        # "weight_decay": [1e-2, 1e-3],
    })),
    # "mlp": list(ParameterGrid({
    #     "hidden_dims": [(64, 32)],
    #     "dropout": [0.0],
    #     "lr_backbone": [1e-4],
    #     "lr_head": [1e-3, 3e-4],
    #     "weight_decay": [1e-2, 1e-3],
    # })),
}

## Load Data

In [3]:
samples, image_paths = load_image_paths(CSV_PATH, TOP_FOLDER, SIDE_FOLDER)
print("Samples shape:", samples.shape)
display(samples[["exp_id", "volume", "top_path", "side_path"]].head(8))

Samples shape: (420, 24)


,exp_id,volume,top_path,side_path
0,1,1.06,photos/top_view_images/P2260331.JPG,photos/side_view_images/P2260685.JPG
1,1,2.12,photos/top_view_images/P2260332.JPG,photos/side_view_images/P2260686.JPG
2,1,3.18,photos/top_view_images/P2260333.JPG,photos/side_view_images/P2260687.JPG
3,1,4.24,photos/top_view_images/P2260334.JPG,photos/side_view_images/P2260688.JPG
4,2,1.06,photos/top_view_images/P2260335.JPG,photos/side_view_images/P2260689.JPG
5,2,2.12,photos/top_view_images/P2260336.JPG,photos/side_view_images/P2260690.JPG
6,2,3.18,photos/top_view_images/P2260337.JPG,photos/side_view_images/P2260692.JPG
7,2,4.24,photos/top_view_images/P2260339.JPG,photos/side_view_images/P2260693.JPG


## Show Regression Model Configurations

In [4]:
print("Grid sizes per head:")
for name, grid in LAST_LAYER_REGRESSION_MODEL_CONFIGS.items():
    n = len(grid)
    print(f"  {name:10s}: {n:4d}")

Grid sizes per head:
  linear    :    1


## Nested CV Evaluation (split by experiment IDs)

In [5]:
all_results = []
nested_artifacts = {}

for backbone_name in LAST_LAYER_BACKBONE_NAMES:
    for fusion_name in LAST_LAYER_FUSION_NAMES:
        print(f"\n{'#'*90}")
        print(f"Evaluating backbone={backbone_name}, fusion={fusion_name}")
        print(f"{'#'*90}")

        nested_results, oof_predictions, training_histories = run_end_to_end_nested_cv(
            samples_df=samples,
            backbone_name=backbone_name,
            fusion_name=fusion_name,
            head_configs=LAST_LAYER_REGRESSION_MODEL_CONFIGS,
            outer_splits=LAST_LAYER_OUTER_SPLITS,
            inner_splits=LAST_LAYER_INNER_SPLITS,
            batch_size=BATCH_SIZE,
            max_epochs=MAX_EPOCHS,
            device=DEVICE,
            side_mask_paths=SIDE_ROI_MASKS,
            top_mask_paths=TOP_ROI_MASKS,
            patience=PATIENCE,
            min_delta=0.0
        )

        summary_df = summarise_nested_results(nested_results, backbone_name, fusion_name)
        all_results.append(summary_df)
        nested_artifacts[(backbone_name, fusion_name)] = {
            "y": samples["volume"].to_numpy(dtype=float),
            "groups": samples["exp_id"].to_numpy(),
            "nested_results": nested_results,
            "oof_predictions": oof_predictions,
            "training_histories": training_histories,
        }

results_df = pd.concat(all_results, ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean", "cv_r2_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)


##########################################################################################
Evaluating backbone=densenet121, fusion=concat
##########################################################################################

GroupKFold | densenet121 | concat | linear


Outer Folds:   0%|          | 0/3 [00:00<?, ?it/s]
# Configs | Outer Fold 1/3:   0%|          | 0/1 [00:00<?, ?it/s]

## Inner Folds | Outer Fold 1/3 | Config 1/1:   0%|          | 0/3 [00:00<?, ?it/s]


### Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

    epoch=1, batch=1/12, loss=6.9610
    epoch=1, batch=2/12, loss=2.1919
    epoch=1, batch=3/12, loss=1.6754
    epoch=1, batch=4/12, loss=0.9668
    epoch=1, batch=5/12, loss=1.7480
    epoch=1, batch=6/12, loss=2.0368
    epoch=1, batch=7/12, loss=0.9651
    epoch=1, batch=8/12, loss=1.8125
    epoch=1, batch=9/12, loss=1.0002
    epoch=1, batch=10/12, loss=0.5092
    epoch=1, batch=11/12, loss=0.5639
    epoch=1, batch=12/12, loss=0.6685





### Epochs:   0%|          | 0/1 [02:18<?, ?it/s, best_val_mae=inf, train_loss=1.7583, val_mae=0.4801]


### Epochs: 100%|██████████| 1/1 [02:18<00:00, 138.48s/it, best_val_mae=inf, train_loss=1.7583, val_mae=0.4801]


                                                                                                               

## Inner Folds | Outer Fold 1/3 | Config 1/1:  33%|███▎      | 1/3 [02:18<04:37, 138.61s/it]


### Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

    epoch=1, batch=1/12, loss=7.5520
    epoch=1, batch=2/12, loss=6.0282
    epoch=1, batch=3/12, loss=1.7861
    epoch=1, batch=4/12, loss=1.1927
    epoch=1, batch=5/12, loss=1.4500
    epoch=1, batch=6/12, loss=1.7868
    epoch=1, batch=7/12, loss=2.3836
    epoch=1, batch=8/12, loss=2.0902
    epoch=1, batch=9/12, loss=4.3930
    epoch=1, batch=10/12, loss=0.9445
    epoch=1, batch=11/12, loss=1.4093
    epoch=1, batch=12/12, loss=0.8589


...

Outer Fold 1: MAE=0.444, RMSE=0.600, R2=0.7409443105891171, best={'lr_backbone': 0.0001, 'lr_head': 0.001, 'weight_decay': 0.001}



# Configs | Outer Fold 2/3:   0%|          | 0/1 [00:00<?, ?it/s]

## Inner Folds | Outer Fold 2/3 | Config 1/1:   0%|          | 0/3 [00:00<?, ?it/s]


### Epochs:   0%|          | 0/1 [00:00<?, ?it/s]

    epoch=1, batch=1/12, loss=10.3017
    epoch=1, batch=2/12, loss=7.3828
    epoch=1, batch=3/12, loss=0.9774
    epoch=1, batch=4/12, loss=1.0831
    epoch=1, batch=5/12, loss=1.6275
    epoch=1, batch=6/12, loss=2.6803
    epoch=1, batch=7/12, loss=2.8510
    epoch=1, batch=8/12, loss=1.8744
    epoch=1, batch=9/12, loss=1.9377
    epoch=1, batch=10/12, loss=1.8897





                                                 

                                                                                   
Outer Folds:  33%|███▎      | 1/3 [13:22<26:44, 802.26s/it, MAE=0.444, RMSE=0.600, best={'lr_backbone': 0.0001, 'lr_head': 0.001, 'weight_decay': 0.001}]


KeyboardInterrupt: 

## Add Dummy Mean Predictions for Comparison

In [ ]:
dummy_pred = np.full(len(samples), samples["volume"].mean(), dtype=float)
dummy_mae = mean_absolute_error(samples["volume"], dummy_pred)
dummy_mse = mean_squared_error(samples["volume"], dummy_pred)
dummy_rmse = np.sqrt(dummy_mse)
dummy_r2 = r2_score(samples["volume"], dummy_pred)

dummy_row = pd.DataFrame([{
    "backbone": "dummy_mean",
    "fusion": "dummy_mean",
    "regressor": "dummy_mean",
    "cv_mae_mean": dummy_mae,
    "cv_mae_std": 0.0,
    "cv_mse_mean": dummy_mse,
    "cv_mse_std": 0.0,
    "cv_rmse_mean": dummy_rmse,
    "cv_rmse_std": 0.0,
    "cv_r2_mean": dummy_r2,
    "cv_r2_std": 0.0,
}])

results_df = pd.concat([results_df, dummy_row], ignore_index=True).sort_values(
    ["cv_mae_mean", "cv_rmse_mean", "cv_r2_mean"],
    ascending=[True, True, False],
).reset_index(drop=True)

## Show Selected Hyperparameters

In [ ]:
for (backbone_name, fusion_name), artifact in nested_artifacts.items():
    print(f"\n{'='*90}")
    print(f"backbone={backbone_name}, fusion={fusion_name}")
    print(f"{'='*90}")
    for head_name, folds in artifact["nested_results"].items():
        has_params = any(f["best_params"] for f in folds)
        if not has_params:
            continue
        print(head_name)
        for f in folds:
            print(f"  Fold {f['fold']}: {f['best_params']}")
        print()


## Show Performance for All Configurations
Prints mean and standard deviation of MAE, MSE, RMSE, and R2 over the CV folds.
Plots out-of-fold predictions for the best configurations.


In [ ]:
display(results_df)

for _, row in results_df.head(5).iterrows():
    key = (row["backbone"], row["fusion"])
    if key in nested_artifacts:
        artifact = nested_artifacts[key]
        y_true = artifact["y"]
        y_pred = artifact["oof_predictions"][row["regressor"]]
        title = f"{row['backbone']} | {row['fusion']} | {row['regressor']}"
        make_oof_plot(y_true, y_pred, title_prefix=title)

## Save Results

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
name = f"top_side_last_stage_unfrozen_{timestamp}.csv"
results_df.to_csv(OUTPUT_DIR / name, index=False)
print("Saved:", OUTPUT_DIR / name)

hyperparams_name = f"top_last_stage_unfrozen_best_hyperparameters_{timestamp}.json"

best_hyperparameters = {}
for (backbone_name, fusion_name), artifact in nested_artifacts.items():
    config_key = f"{backbone_name}__{fusion_name}"
    best_hyperparameters[config_key] = {}
    for regressor_name, folds in artifact["nested_results"].items():
        fold_params = [
            {
                "fold": f["fold"],
                "best_params": f["best_params"],
                "MAE": f.get("MAE"),
                "RMSE": f.get("RMSE"),
                "R2": f.get("R2"),
                "MSE": f.get("MSE"),
            }
            for f in folds
        ]

        best_hyperparameters[config_key][regressor_name] = fold_params

with open(OUTPUT_DIR / hyperparams_name, "w") as f:
    json.dump(best_hyperparameters, f, indent=2, default=str)

print("Saved:", OUTPUT_DIR / hyperparams_name)

oof_rows = []
sample_meta = samples[["file_name", "exp_id", "volume", "top_path", "side_path"]].reset_index(drop=True)

for (backbone_name, fusion_name), artifact in nested_artifacts.items():
    for regressor_name, y_pred in artifact["oof_predictions"].items():
        y_true = artifact["y"]
        for i, (yt, yp) in enumerate(zip(y_true, y_pred)):
            oof_rows.append({
                "backbone": backbone_name,
                "fusion": fusion_name,
                "regressor": regressor_name,
                "sample_idx": i,
                "exp_id": sample_meta.loc[i, "exp_id"],
                "top_path": sample_meta.loc[i, "top_path"],
                "side_path": sample_meta.loc[i, "side_path"],
                "volume": sample_meta.loc[i, "volume"],
                "y_true": float(yt),
                "y_pred": float(yp),
                "residual": float(yp - yt),
                "abs_error": float(abs(yp - yt)),
            })

oof_df = pd.DataFrame(oof_rows)
oof_name = f"top_side_last_stage_unfrozen_oof_predictions_{timestamp}.csv"
oof_df.to_csv(OUTPUT_DIR / oof_name, index=False)

print("Saved:", OUTPUT_DIR / oof_name)